# IRIS: 01 Prepare data

Turns the four raw datasets into the small files the dashboard reads.

| Output (in `data/processed/`) | What it holds |
| --- | --- |
| `province_index.csv` | One row per province: hazard, exposure, Flood-Health Exposure Index, rank, robustness |
| `dekad_rain.csv` | 10-day rainfall per province since 2015 with each province's own 90th percentile, for the rainfall trigger |
| `weekly_national.csv` | Weekly national rainfall indicators joined to weekly leptospirosis cases, for the lag analysis |
| `provinces.geojson` | Simplified province shapes keyed by `PCODE` (optional step, needs the boundary geodatabase) |

**Province units.** The rainfall file uses 81 units with codes like `PH01028`. Census barangays and the
2025 boundary file are mapped onto those 81 units. NCR is one unit (`PH13074`), Davao Occidental is merged
into Davao del Sur, Maguindanao del Norte and del Sur plus the Special Geographic Area are one Maguindanao
unit, and City of Isabela is counted with Basilan. These match the rainfall file's pixel counts.

Run top to bottom. On Colab, run the setup cell first and point `RAW` at the uploaded `datasets` folder.

In [1]:
# Setup. On Colab, uncomment the Drive lines and set RAW to the datasets folder.
# from google.colab import drive
# drive.mount('/content/drive')
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path('..') if Path.cwd().name == 'notebooks' else Path('.')   # project folder
RAW = ROOT / 'datasets'          # on Colab, e.g. Path('/content/drive/MyDrive/IRIS/datasets')
OUT = ROOT / 'data' / 'processed'
OUT.mkdir(parents=True, exist_ok=True)

RAIN_CSV = RAW / 'phl-rainfall-subnat-full.csv'
CENSUS_XLSX = RAW / '2020-census-total-popn_single-age_malefemale_admin4.xlsx'
DISEASE_XLSX = RAW / 'Diseases Data (for NBL).xlsx'
BOUNDARY_GDB = RAW / 'phl_admin_boundaries.gdb'   # folder, or the downloaded .gdb.zip

BASELINE = (1981, 2024)   # full years used for percentiles and thresholds
TRIGGER_FROM = '2015-01-01'
N_RUNS = 1000
SEED = 2026

## 1. Rainfall

Keep the province rows (`adm_level == 2`). `rfh` is the 10-day (dekad) rainfall total in mm and `r1h` is the
rolling 1-month total, which stands in for the 30-day total in the proposal. The last two dekads are marked
`prelim` by the publisher; they are kept for the trigger but left out of the baseline.

In [2]:
rain = pd.read_csv(RAIN_CSV, parse_dates=['date'])
rain = rain[rain['adm_level'] == 2][['date', 'PCODE', 'n_pixels', 'rfh', 'rfh_avg', 'r1h', 'version']].copy()
UNITS = sorted(rain['PCODE'].unique())
print(len(UNITS), 'province units;', rain['date'].min().date(), 'to', rain['date'].max().date())
base = rain[rain['date'].dt.year.between(*BASELINE) & (rain['version'] == 'final')]

81 province units; 1981-01-01 to 2026-10-11


## 2. Hazard score

A note on the proposal's measure (a): the share of dekads above a province's **own** 90th percentile is about
10% for every province by definition, so it cannot rank provinces. Here it uses one **national** threshold
instead: the 90th percentile of all province dekads in the baseline. Measure (b) is the 95th percentile of the
rolling 1-month total. Each is turned into a percentile rank across provinces and the two are averaged into a
0 to 100 score, for the full year and for June to November.

In [3]:
def pct_rank(s):
    return s.rank(pct=True) * 100

NAT_P90 = base['rfh'].quantile(0.90)
print(f'National 90th percentile of 10-day rainfall: {NAT_P90:.1f} mm')

def hazard(df):
    g = df.groupby('PCODE')
    share = g['rfh'].apply(lambda s: (s > NAT_P90).mean())
    r1h_p95 = g['r1h'].quantile(0.95)
    return pd.DataFrame({'share_extreme': share, 'r1h_p95': r1h_p95,
                         'score': (pct_rank(share) + pct_rank(r1h_p95)) / 2})

hz_year = hazard(base)
hz_wet = hazard(base[base['date'].dt.month.between(6, 11)])
hz = pd.DataFrame({
    'share_extreme_dekads': hz_year['share_extreme'],
    'rain_30d_p95_mm': hz_year['r1h_p95'],
    'hazard_year': hz_year['score'],
    'hazard_jun_nov': hz_wet['score'],
})
hz['hazard'] = hz[['hazard_year', 'hazard_jun_nov']].mean(axis=1)
hz.sort_values('hazard', ascending=False).head()

National 90th percentile of 10-day rainfall: 163.2 mm


,share_extreme_dekads,rain_30d_p95_mm,hazard_year,hazard_jun_nov,hazard
PCODE,,,,,
PH06006,0.230429,746.298285,96.913580,98.148148,97.530864
PH03071,0.177399,809.815840,95.679012,98.765432,97.222222
PH03008,0.169192,821.188032,94.753086,98.765432,96.759259
PH01033,0.152778,753.047830,89.506173,95.679012,92.592593
PH05016,0.190025,745.073975,93.827160,84.567901,89.197531


## 3. Census exposure

Barangays are keyed by the first 7 characters of `BgyCode_new` (for example `PH01028`). Highly urbanized and
independent cities carry their own codes, so they are mapped back to their province by name, and the merged
units listed at the top are mapped by hand. Every barangay must land in one of the 81 units.

In [4]:
census = pd.read_excel(CENSUS_XLSX, engine='openpyxl')
age = [c for c in census.columns if c.endswith('_MF')]
census[age] = census[age].replace('-', 0).apply(pd.to_numeric, errors='coerce').fillna(0)
census = census.copy()   # de-fragment the 250-column frame

U5 = ['Under 1_MF', '1_MF', '2_MF', '3_MF', '4_MF']
O65 = [f'{a}_MF' for a in range(65, 80)] + ['80 and Over_MF']
census['under5'] = census[U5].sum(axis=1)
census['over65'] = census[O65].sum(axis=1)
census['key'] = census['BgyCode_new'].str[:7]

MANUAL = {
    'PH09901': 'PH19007',   # City of Isabela -> Basilan
    'PH11086': 'PH11024',   # Davao Occidental -> Davao del Sur (pre-2013 unit)
    'PH19038': 'PH19087',   # Maguindanao -> Maguindanao unit
    'PH19999': 'PH19087',   # Special Geographic Area -> Maguindanao unit
}
def to_unit(row):
    k = row['key']
    if k in UNITS:
        return k
    if k.startswith('PH138'):
        return 'PH13074'    # all of NCR
    return MANUAL.get(k)
census['PCODE'] = census.apply(to_unit, axis=1)

# Cities with their own code: use the province code most barangays of that province name use.
main = (census.dropna(subset=['PCODE']).groupby('Province')['PCODE']
        .agg(lambda s: s.value_counts().index[0]))
census['PCODE'] = census['PCODE'].fillna(census['Province'].map(main))

missing = census[census['PCODE'].isna()]
assert missing.empty, missing[['Province', 'key']].drop_duplicates()
print('Barangays:', len(census), '| total population:', int(census['Total_MF'].sum()))
census.loc[census['key'] != census['PCODE'], ['key', 'Province', 'PCODE']].drop_duplicates('key')

Barangays: 42041 | total population: 109033245


,key,Province,PCODE
0,PH13806,"NCR, City of Manila, First District (Not a Pro...",PH13074
897,PH13805,"NCR, Second District (Not a Province)",PH13074
924,PH13807,"NCR, Second District (Not a Province)",PH13074
940,PH13812,"NCR, Second District (Not a Province)",PH13074
970,PH13813,"NCR, Second District (Not a Province)",PH13074
1112,PH13814,"NCR, Second District (Not a Province)",PH13074
1133,PH13801,"NCR, Third District (Not a Province)",PH13074
1321,PH13804,"NCR, Third District (Not a Province)",PH13074
1342,PH13809,"NCR, Third District (Not a Province)",PH13074
1360,PH13816,"NCR, Third District (Not a Province)",PH13074


In [5]:
names = (census.groupby('PCODE')['Province']
         .agg(lambda s: s.value_counts().index[0]).str.title()
         .str.replace(' (Not A Province)', '', regex=False)
         .str.replace(r'\bDel\b', 'del', regex=True).str.replace(r'\bDe\b', 'de', regex=True))
names['PH13074'] = 'Metro Manila (NCR)'
names['PH19087'] = 'Maguindanao'
pop = census.groupby('PCODE')[['Total_MF', 'under5', 'over65']].sum().rename(columns={'Total_MF': 'population'})
pop['vulnerable'] = pop['under5'] + pop['over65']
pop['vulnerable_share'] = pop['vulnerable'] / pop['population']
pop['exp_count'] = pct_rank(pop['vulnerable'])
pop['exp_share'] = pct_rank(pop['vulnerable_share'])
pop.insert(0, 'province', names)
assert set(pop.index) == set(UNITS)
pop.sort_values('vulnerable', ascending=False).head()

,province,population,under5,over65,vulnerable,vulnerable_share,exp_count,exp_share
PCODE,,,,,,,,
PH13074,Metro Manila (NCR),13484462,1279827,638816,1918643,0.142285,100.000000,4.938272
PH07022,Cebu,5151274,533370,263062,796432,0.154609,98.765432,32.098765
PH04021,Cavite,4344829,401628,201668,603296,0.138854,97.530864,1.234568
PH03014,Bulacan,3708890,346709,192864,539573,0.145481,96.296296,7.407407
PH06045,Negros Occidental,3223955,320013,203637,523650,0.162425,95.061728,60.493827


## 4. Flood-Health Exposure Index and robustness

Baseline weights are 0.4 hazard, 0.3 vulnerable count, 0.3 vulnerable share. The robustness check draws 1,000
random weight sets, each weight at least 0.1 and summing to 1, and records each province's rank range and how
often it stays in the top 10.

In [6]:
df = pop.join(hz)
X = df[['hazard', 'exp_count', 'exp_share']].to_numpy()
W0 = np.array([0.4, 0.3, 0.3])
df['index'] = X @ W0
df['rank'] = df['index'].rank(ascending=False, method='min').astype(int)

rng = np.random.default_rng(SEED)
W = 0.1 + 0.7 * rng.dirichlet([1, 1, 1], size=N_RUNS)      # each >= 0.1, rows sum to 1
ranks = pd.DataFrame(X @ W.T, index=df.index).rank(ascending=False, method='min')
df['rank_best'] = ranks.min(axis=1).astype(int)
df['rank_worst'] = ranks.max(axis=1).astype(int)
df['top10_pct'] = (ranks <= 10).mean(axis=1) * 100
df['confidence'] = np.where(df['top10_pct'] >= 80, 'High',
                   np.where(df['top10_pct'] >= 40, 'Medium', np.where(df['rank'] <= 10, 'Low', '')))
df = df.sort_values('rank')
df.round(3).to_csv(OUT / 'province_index.csv', index_label='PCODE')
df[['province', 'index', 'rank', 'rank_best', 'rank_worst', 'top10_pct', 'confidence']].head(15)

,province,index,rank,rank_best,rank_worst,top10_pct,confidence
PCODE,,,,,,,
PH01055,Pangasinan,80.493827,1,1,20,95.7,High
PH05017,Camarines Sur,78.703704,2,1,11,99.8,High
PH06006,Antique,77.160494,3,1,43,74.9,Medium
PH05016,Camarines Norte,72.345679,4,2,43,60.4,Medium
PH03069,Tarlac,69.753086,5,3,34,55.8,Medium
PH05020,Catanduanes,68.765432,6,1,60,52.4,Medium
PH19036,Lanao del Sur,68.641975,7,3,28,58.0,Medium
PH06030,Iloilo,67.777778,8,3,34,57.4,Medium
PH08037,Leyte,66.481481,9,4,38,47.4,Medium


## 5. Trigger table for the dashboard

Each province's own 90th percentile of 10-day rainfall (baseline years) is the alert threshold. The table keeps
dekads from 2015 on so the app stays small.

In [7]:
p90 = base.groupby('PCODE')['rfh'].quantile(0.90).rename('p90_mm')
trig = rain[rain['date'] >= TRIGGER_FROM].join(p90, on='PCODE')
trig['above_p90'] = trig['rfh'] > trig['p90_mm']
trig = trig[['date', 'PCODE', 'rfh', 'rfh_avg', 'p90_mm', 'above_p90', 'version']].round({'rfh': 2, 'rfh_avg': 2, 'p90_mm': 2})
trig.to_csv(OUT / 'dekad_rain.csv', index=False)
print(len(trig), 'rows; latest dekad', trig['date'].max().date(),
      '| provinces above p90 then:', int(trig.loc[trig['date'] == trig['date'].max(), 'above_p90'].sum()))

34425 rows; latest dekad 2026-10-11 | provinces above p90 then: 2


## 6. Weekly national series for the lag analysis

Each dekad total is spread evenly over its days (dekad 3 runs 8 to 11 days), summed into ISO weeks per
province, and then summarized nationally: the share of provinces above their own weekly 90th percentile and the
mean weekly rainfall. Weekly leptospirosis counts (2019 to 2023 in this file) are reshaped to year and week and
joined on ISO year and week, which approximates DOH morbidity weeks. Missing weeks stay missing.

In [8]:
r = rain[['date', 'PCODE', 'rfh']].copy()
nxt = r['date'] + pd.offsets.MonthBegin(1)
r['end'] = np.where(r['date'].dt.day == 21, nxt, r['date'] + pd.Timedelta(days=10))
r['ndays'] = (pd.to_datetime(r['end']) - r['date']).dt.days
daily = r.loc[r.index.repeat(r['ndays'])].copy()
daily['day'] = daily['date'] + pd.to_timedelta(daily.groupby(level=0).cumcount(), unit='D')
daily['mm'] = daily['rfh'] / daily['ndays']
iso = daily['day'].dt.isocalendar()
daily['iso_year'], daily['iso_week'] = iso['year'].astype(int), iso['week'].astype(int)

wk = daily.groupby(['PCODE', 'iso_year', 'iso_week'])['mm'].agg(['sum', 'size']).reset_index()
wk = wk[wk['size'] == 7]                                   # drop partial weeks at the ends
wk_p90 = wk[wk['iso_year'].between(*BASELINE)].groupby('PCODE')['sum'].quantile(0.90)
wk_p90.rename('weekly_p90_mm').round(2).to_csv(OUT / 'province_weekly_p90.csv')   # used by the live forecast
wk['above'] = wk['sum'] > wk['PCODE'].map(wk_p90)
nat = wk.groupby(['iso_year', 'iso_week']).agg(
    share_prov_above_p90=('above', 'mean'), mean_rain_mm=('sum', 'mean')).reset_index()

dis = pd.read_excel(DISEASE_XLSX)
lep = [c for c in dis.columns if c.lower().startswith('leptospirosis')]
lepto = (dis[['Week'] + lep].dropna(subset=['Week'])
         .melt(id_vars='Week', var_name='col', value_name='cases'))
lepto['iso_year'] = lepto['col'].str[-4:].astype(int)
lepto['iso_week'] = lepto['Week'].astype(int)
lepto = lepto[['iso_year', 'iso_week', 'cases']]

weekly = nat.merge(lepto, on=['iso_year', 'iso_week'], how='left')
weekly = weekly[weekly['iso_year'] >= lepto['iso_year'].min() - 1]
weekly['week_start'] = pd.to_datetime(
    weekly['iso_year'].astype(str) + weekly['iso_week'].astype(str).str.zfill(2) + '1', format='%G%V%u')
weekly['covid_period'] = weekly['iso_year'].isin([2020, 2021])
weekly.round({'share_prov_above_p90': 4, 'mean_rain_mm': 2}).to_csv(OUT / 'weekly_national.csv', index=False)
print(weekly['cases'].notna().sum(), 'weeks with case counts;',
      weekly.groupby('iso_year')['cases'].count().to_dict())

227 weeks with case counts; {2018: 0, 2019: 52, 2020: 28, 2021: 52, 2022: 52, 2023: 43, 2024: 0, 2025: 0, 2026: 0}


## 7. Province shapes (optional)

Skip this if `provinces.geojson` is already in `data/processed/`. It reads only the province layer of the
boundary geodatabase, merges the 2025 units into the 81 rainfall units and simplifies the shapes to about 1 MB.
Needs `geopandas` (preinstalled on Colab).

In [9]:
RUN_GEO = not (OUT / 'provinces.geojson').exists()
if RUN_GEO:
    import geopandas as gpd
    shp = gpd.read_file(BOUNDARY_GDB, layer='phl_admin2', columns=['adm2_pcode'])
    MERGE = {'PH13039': 'PH13074', 'PH13075': 'PH13074', 'PH13076': 'PH13074', 'PH09097': 'PH19007',
             'PH11086': 'PH11024', 'PH19088': 'PH19087', 'PH19099': 'PH19087'}
    shp['PCODE'] = shp['adm2_pcode'].replace(MERGE)
    assert set(shp['PCODE']) == set(UNITS)
    shp = shp.dissolve('PCODE').reset_index()[['PCODE', 'geometry']].to_crs(4326)
    shp['geometry'] = shp.simplify(0.01, preserve_topology=True)
    shp.to_file(OUT / 'provinces.geojson', driver='GeoJSON')
print(sorted(p.name for p in OUT.iterdir()))

['dekad_rain.csv', 'disease_lag.csv', 'event_study.csv', 'forecast_backtest.csv', 'forecast_coefs.csv', 'forecast_scores.csv', 'lag_curve.png', 'lag_results.csv', 'ncr_barangays.geojson', 'ncr_flood_events.csv', 'ncr_month_outlook.csv', 'province_index.csv', 'province_points.csv', 'province_weekly_p90.csv', 'provinces.geojson', 'weekly_case_baseline.csv', 'weekly_national.csv']
